# 🤖 Agent Tool-Calling Loop — Guided Demo

**DevRev Technical Round · Section 3.** A runnable walkthrough of a DevRev-flavored **ReAct agent**:
query → pick a tool → observe → loop → answer, with a **max-iteration guard**, **confirmation gate**,
**memoization**, **retry**, **fallback**, **disambiguation**, and **observability** — built both
**from scratch** and with **LangGraph**.

> No API key needed: the "brain" is a deterministic rule-based router that mimics an LLM's tool
> selection, so every cell below is reproducible offline.

## 0. Setup — make the `src/` package importable

In [1]:
import sys, os
# Find the project root (the folder that contains `src/`), whether we launched from
# notebooks/ or from the project root.
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "src")):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

from src.tools import TicketStore, build_registry
from src.brain import RuleBasedBrain
from src.scratch_agent import Agent, Session
from src.robustness import (always_approve, deny_destructive, execute_tool,
                            with_fallback, disambiguate)
from src.observability import ToolCallLogger
print("imports ok — project root:", _root)

imports ok — project root: d:\INTERVIEW PREPARATION\DevRev_Preparation\agent_tool_calling_demo


## 1. The tool registry

A **tool** = name + description (what the model reads to choose it) + the function + a
`destructive` flag. The registry is what you'd hand an LLM so it knows what it can call.

In [2]:
store = TicketStore()
registry = build_registry(store)
for spec in registry.specs():
    flag = "  ⚠️ destructive" if spec["destructive"] else ""
    print(f"- {spec['name']:<15} {spec['description']}{flag}")

- search_tickets  Find open tickets whose subject or tags match a query string. Args: query (str).
- get_ticket      Fetch full details of one ticket by its id. Args: ticket_id (str).
- create_ticket   Create a new ticket with a subject. Args: subject (str).
- close_ticket    Close (resolve) a ticket by id. DESTRUCTIVE — needs confirmation. Args: ticket_id (str).  ⚠️ destructive


## 2. The ReAct loop (from scratch) — a single-step query

`Agent.run` loops: **think** (brain picks a tool or finishes) → **act** (execute) →
**observe** → repeat, until a final answer or the max-iteration guard.

In [3]:
agent = Agent(registry, confirm=always_approve)   # auto-approve for the happy path
sess = Session()
result = agent.run("find tickets about auth", sess)
print("ANSWER:", result.answer)
print("iterations:", result.iterations)
print("\nTRACE:")
print(sess.logger.trace())

ANSWER: Found 2 ticket(s): TKT-1, TKT-3.
iterations: 1

TRACE:
 #  tool             status         ms  args -> result/error
------------------------------------------------------------------------------
 1  search_tickets   ok            0.0  {'query': 'auth'} -> [{'id': 'TKT-1', 'subject': 'Cannot log…


## 3. Multi-step + the confirmation gate

"close the ticket about login" needs **two** tool calls: `search_tickets` then
`close_ticket`. Because `close_ticket` is **destructive**, the agent pauses for
confirmation. First we deny (nothing happens), then we approve.

In [4]:
# 3a. Default policy DENIES destructive actions -> the agent pauses.
store_a = TicketStore(); reg_a = build_registry(store_a)
blocked = Agent(reg_a, confirm=deny_destructive).run("close the ticket about login")
print("BLOCKED:", blocked.answer)
print("blocked_on:", blocked.blocked_on)
print("ticket still open? ->", store_a.tickets["TKT-3"]["status"])

BLOCKED: Awaiting confirmation to run 'close_ticket' with {'ticket_id': 'TKT-3'}.
blocked_on: ('close_ticket', {'ticket_id': 'TKT-3'})
ticket still open? -> open


In [5]:
# 3b. With approval, the same request completes.
store_b = TicketStore(); reg_b = build_registry(store_b); sess_b = Session()
done = Agent(reg_b, confirm=always_approve).run("close the ticket about login", sess_b)
print("ANSWER:", done.answer, "| iterations:", done.iterations)
print("ticket status now ->", store_b.tickets["TKT-3"]["status"])
print("\nTRACE:")
print(sess_b.logger.trace())

ANSWER: Closed TKT-3 — 'Login page 500 error'. | iterations: 2
ticket status now -> closed

TRACE:
 #  tool             status         ms  args -> result/error
------------------------------------------------------------------------------
 1  search_tickets   ok            0.0  {'query': 'login'} -> [{'id': 'TKT-3', 'subject': 'Login page…
 2  close_ticket     ok            0.0  {'ticket_id': 'TKT-3'} -> {'id': 'TKT-3', 'subject': 'Login page …


## 4. Robustness — retry on a flaky tool

A transient failure (network blip / 503) should be **retried**, not surfaced. Here
`search_tickets` fails its first two calls, then succeeds; the executor retries.

In [6]:
log = ToolCallLogger()
flaky_reg = build_registry(TicketStore(), flaky_search=True)   # fails twice, then works
res = execute_tool(flaky_reg.get("search_tickets"), {"query": "auth"}, logger=log, retries=3)
print("result:", [h["id"] for h in res])
print("retries:", log.count("retry"), "| final ok:", log.count("ok"))
print("\nTRACE:")
print(log.trace())

result: ['TKT-1', 'TKT-3']
retries: 2 | final ok: 1

TRACE:
 #  tool             status         ms  args -> result/error
------------------------------------------------------------------------------
 1  search_tickets   retry         0.0  {'query': 'auth'} -> temporary outage (attempt 1)
 2  search_tickets   retry         0.0  {'query': 'auth'} -> temporary outage (attempt 2)
 3  search_tickets   ok            0.0  {'query': 'auth'} -> [{'id': 'TKT-1', 'subject': 'Cannot log…


## 5. Robustness — memoization

An identical `(tool, args)` call within a session is served from cache instead of
re-hitting the backend. Watch the second call become a `cache_hit`.

In [7]:
log = ToolCallLogger(); memo = {}
for _ in range(2):
    execute_tool(registry.get("search_tickets"), {"query": "billing"}, logger=log, memo=memo)
print("ok:", log.count("ok"), "| cache_hit:", log.count("cache_hit"))
print(log.trace())

ok: 1 | cache_hit: 1
 #  tool             status         ms  args -> result/error
------------------------------------------------------------------------------
 1  search_tickets   ok            0.0  {'query': 'billing'} -> [{'id': 'TKT-2', 'subject': 'Billing ov…
 2  search_tickets   cache_hit     0.0  {'query': 'billing'} -> [{'id': 'TKT-2', 'subject': 'Billing ov…


## 6. Robustness — fallback

If a permanent failure happens (`get_ticket` on a missing id), fall back to an
alternative tool (`search_tickets`), optionally re-mapping the arguments.

In [8]:
log = ToolCallLogger()
out = with_fallback(
    registry.get("get_ticket"), registry.get("search_tickets"),
    {"ticket_id": "TKT-999"},                 # doesn't exist -> permanent failure
    adapt=lambda a: {"query": "login"},        # re-map args for the fallback tool
    logger=log,
)
print("fallback result:", [h["id"] for h in out])
print(log.trace())

fallback result: ['TKT-3']
 #  tool             status         ms  args -> result/error
------------------------------------------------------------------------------
 1  get_ticket       error         0.0  {'ticket_id': 'TKT-999'} -> ticket 'TKT-999' does not exist
 2  get_ticket       error         0.0  {'ticket_id': 'TKT-999'} -> falling back to search_tickets
 3  search_tickets   ok            0.0  {'query': 'login'} -> [{'id': 'TKT-3', 'subject': 'Login page…


## 7. Robustness — disambiguation

When two tools could plausibly answer the same query, choose deterministically and
**never silently run a destructive tool on a tie** — safe tools win.

In [9]:
candidates = [registry.get("search_tickets"), registry.get("close_ticket")]
picked = disambiguate(candidates, "find and close tickets about auth")
print("candidates:", [t.name for t in candidates])
print("picked:", picked.name, "(non-destructive preferred)")

candidates: ['search_tickets', 'close_ticket']
picked: search_tickets (non-destructive preferred)


## 8. State across conversation turns

A `Session` persists **history + memo + logger** across turns. A repeated search in a
later turn is a free cache hit.

In [10]:
conv = Session()
a = Agent(registry, confirm=always_approve)
print(a.run("find tickets about auth", conv).answer)
print(a.run("find tickets about auth", conv).answer)   # identical -> cached
print("\ncache hits this session:", conv.logger.count("cache_hit"))

Found 2 ticket(s): TKT-1, TKT-3.
Found 2 ticket(s): TKT-1, TKT-3.

cache hits this session: 1


## 9. Observability — the full session trace

Everything the agent did, in order: tool, status (ok/retry/cache_hit/blocked/error),
latency, args → result. This is what you'd ship to LangSmith / OpenTelemetry.

In [11]:
print(conv.logger.trace())

 #  tool             status         ms  args -> result/error
------------------------------------------------------------------------------
 1  search_tickets   ok            0.0  {'query': 'auth'} -> [{'id': 'TKT-1', 'subject': 'Cannot log…
 2  search_tickets   cache_hit     0.0  {'query': 'auth'} -> [{'id': 'TKT-1', 'subject': 'Cannot log…


## 10. The same loop with **LangGraph**

Two nodes — `agent` (think) and `tools` (act) — with a conditional edge that loops while
there are tool calls and we're under the iteration budget. The rule-based brain emits an
`AIMessage` with `tool_calls` (the exact shape a real LLM with `bind_tools` returns), so
this runs with no API key.

In [12]:
from src.langgraph_agent import build_graph, run_query
store_lg = TicketStore(); reg_lg = build_registry(store_lg); sess_lg = Session()
app = build_graph(reg_lg, sess_lg, confirm=always_approve)

print(run_query(app, "close the ticket about billing"))
print(run_query(app, "find tickets about auth"))
print("billing ticket status ->", store_lg.tickets["TKT-2"]["status"])

Closed TKT-2 — 'Billing overcharge'.
Found 2 ticket(s): TKT-1, TKT-3.
billing ticket status -> closed


### The graph LangGraph compiled (rendered from `app.get_graph()`)

```mermaid
graph TD;
    __start__ --> agent;
    agent -.->|has tool calls & under budget| tools;
    agent -.->|final answer / out of budget| __end__;
    tools --> agent;
```

In [ ]:
# LangGraph can print its own mermaid source:
print(app.get_graph().draw_mermaid())

## 11. Recap — what to say in the interview

1. **The loop:** think → act → observe → repeat, with a **max-iteration guard** so it never runs forever.
2. **Routing:** the brain (LLM or rules) maps intent → the right tool from a registry.
3. **Robustness:** retry transient failures, fall back on permanent ones, escalate if neither works; **memoize** repeats; **disambiguate** ties toward safe tools; **gate destructive actions** behind confirmation.
4. **State & observability:** a `Session` carries history + cache + a trace across turns.
5. **Latency:** run **independent** tool calls in parallel (latency = max, not sum), but keep **dependent** steps sequential and coordinate parallel calls through one rate limiter.

See `docs/DESIGN.md` for the design tradeoffs and `../tutorials/03_Agent_Tool_Calling_Loop.md` for the diagram-driven writeup.